# Chunking
Learn how "garbage in" leads to "garbage out."

## Why do we need chunking?

1. Make sure embedding models can fit the data into their context window  
    -> If you feed a whole book to an embedding model, it will cut off everything past the limit and that data is lost.

2. Make chunks useful for search
    - Reduce hallucinations by giving the model a small, clear set of facts
    - Keep semantic precision. If a chunk covers many topics, its vector averages them and the meaning gets diluted

3. Save cost and reduce latency  
    -> Sending a million-token document to an LLM for every question is very expensive and very slow


In some cases, for long context window of LLM (Like GPT-5.4 and Gemini 3.1 that has 1M context window), un-chunked document will still fit to its context window but
In some cases, for long context window of LLM (Like GPT-5.4 and Gemini 3.1 that has 1M context window), un-chunked document will still fit to its context window but
1. It may increase cost and latency of using the whole context
2. Risk of having [lost in the middle problem](https://medium.com/@cenghanbayram35/lost-in-the-middle-in-llms-86e461dc7212)
3. Providing 3-5 small, highly relevant chunks ensures the AI stays focused on the right information.
4. Your LLM answer won't focus on the topic you specifically ask

## When does Chunking not relevant?

Long-context LLMs (1M+ tokens) can remove the need to chunk for some tasks, but chunking is still useful in many cases.

### When you can skip chunking
- Whole-document reasoning: one-off deep analyses (e.g., themes across a book).
- Short documents: fits the model window — chunking is unnecessary overhead.
- Many-shot prompts: when you must show the model many examples together.

### When you should still chunk
- You use embeddings/vector search (embedding models have smaller input limits).
- You need fast, low-cost, frequent lookups (chunk+retrieve is cheaper than reading the whole doc every time).

**Rule of thumb**: If you need indexed search or repeated queries, chunk. For rare, deep global analysis, whole-document models work well.

# Chunking Strategis

## Fixed-Size Chunking
Simply decide number of tokens in our chunk

In [1]:
from pathlib import Path

# Simple fixed-size token chunking with overlap
# Here, "token" means a whitespace-separated word.
def load_markdown_docs(folder='sample-documents'):
    docs = []
    for path in Path(folder).rglob('*.md'):
        text = path.read_text(encoding='utf-8', errors='ignore').strip()
        docs.append({'id': path.stem, 'path': str(path), 'text': text})
    return docs


def chunk_by_tokens(text, chunk_tokens=120, overlap_tokens=20):
    tokens = text.split() # Split text into tokens (words)
    if len(tokens) <= chunk_tokens: # if text is shorter than chunk size, return as is
        return [' '.join(tokens)]

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_tokens # Calculate end index for the chunk
        chunks.append(' '.join(tokens[start:end])) # Join tokens back into text chunk
        if end >= len(tokens):
            break
        start = max(0, end - overlap_tokens)
    return chunks


docs = load_markdown_docs('sample-documents')
for doc in docs:
    chunks = chunk_by_tokens(doc['text'], chunk_tokens=120, overlap_tokens=20) #set chunk size and overlap here
    print(f"\nFILE: {doc['path']}") # Print file path
    print(f"chunks: {len(chunks)}") # Print number of chunks created
    print(chunks[0][:1000]) # Print first 1000 characters of the first chunk for preview

# you can see the chunk cut points are not very smart, they just split by token count regardless of sentence boundaries or meaning.


FILE: sample-documents\hometown.md
chunks: 8
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags: [hometown, local-history, guide] --- # My Hometown Overview -------- My hometown is a medium-sized city located in the temperate region of the country. It combines a long history with modern amenities and a friendly community. The town is known for its tree-lined streets, a historic downtown area, and a mix of industry, small businesses, and agriculture in the surrounding countryside. Quick facts - Population: ~75,000 (approx.) - Region: Central valley - Founded: 1800s (historic settlement) - Language(s): Primary local language and common secondary languages History ------- The town began as a small trading post in the 19th century and grew with the arrival of the railroad. Early industries

FILE: sample-documents\university.md
chunks: 7
--- title: "My University — A Story of Studying Sustainability" author: "John Doe" source: "personal" create

## Content-aware Chunking
adhere to the structure to help inform the meaning of our chunks

### Simple Sentence and Paragraph Splitting

In [2]:
import re
from pathlib import Path
from typing import List

In [3]:
# Simple sentence & paragraph splitting for Markdown files

# ----------------------
# Paragraph splitting
# ----------------------
def paragraph_split(markdown_text: str) -> List[str]:
    """
    Split markdown into paragraphs by blank lines.
    Keeps paragraphs simple and human-readable.
    """
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n+', markdown_text) if p.strip()]
    return paragraphs

In [4]:
# ----------------------
# Naive sentence split
# ----------------------
_sentence_re = re.compile(r'(?<=[.!?])\s+')

def naive_sentence_split(text: str) -> List[str]:
    """
    Very simple sentence splitter:
    - Splits on punctuation (., !, ?) followed by whitespace.
    - Good for quick demos but will make mistakes on abbreviations.
    """
    sents = [s.strip() for s in _sentence_re.split(text) if s.strip()]
    if not sents:
        # fallback: split on newlines
        sents = [line.strip() for line in text.splitlines() if line.strip()]
    return sents

In [5]:
# ----------------------
# NLTK-based sentence splitting (recommended for many cases)
# ----------------------


def nltk_sentence_split(text: str) -> List[str]:
    """
    Uses NLTK's Punkt sentence tokenizer.
    Install with: pip install nltk
    First run: import nltk; nltk.download('punkt')
    """
    import nltk
    from nltk.tokenize import sent_tokenize

    # [first run only -- uncomment this]
    nltk.download('punkt')

    return sent_tokenize(text)

In [6]:
# ----------------------
# spaCy-based sentence splitting (more robust, needs model)
# ----------------------
def spacy_sentence_split(text: str, model: str = "en_core_web_sm") -> List[str]:
    """
    Uses spaCy for sentence segmentation.
    Install with: pip install spacy
    Then download model: python -m spacy download en_core_web_sm
    spaCy often gives more linguistically-aware splits.
    """
    
    import spacy
    nlp = spacy.load(model)
    
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [7]:
# ----------------------
# Demo: run on all .md files in `rag/sample-documents`
# ----------------------

import textwrap
from pathlib import Path

def _print_header(title: str):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def _print_sub(title: str):
    print("\n" + "-" * 60)
    print(title)
    print("-" * 60)

def _short(s: str, width: int = 160):
    return textwrap.shorten(s.replace("\n", " "), width=width, placeholder=" ...")

def demo_splitters_pretty(folder: str = "sample-documents",
                          max_preview_paras: int = 2,
                          max_preview_sents: int = 3,
                          wrap_width: int = 100):
    """
    Pretty demo that lists paragraphs and sentence-splits (naive / NLTK / spaCy).
    Designed to be readable in notebook outputs.
    """
    p = Path(folder)
    md_files = sorted(p.glob("*.md"))
    if not md_files:
        print(f"No markdown files found in {folder!r}")
        return

    for f in md_files:
        text = f.read_text(encoding="utf8")
        _print_header(f"FILE: {f.name}")

        paras = paragraph_split(text)
        print(f"Paragraphs: {len(paras)} (showing first {min(max_preview_paras, len(paras))})")
        for i, para in enumerate(paras[:max_preview_paras], start=1):
            print(f"\nParagraph {i}:")
            print(textwrap.fill(_short(para, width=400), width=wrap_width))

        # pick a paragraph for sentence demos (prefer first non-empty)
        first_para = next((p for p in paras if p.strip()), text)

        _print_sub("Naive sentence split")
        naive_sents = naive_sentence_split(first_para)
        for i, s in enumerate(naive_sents[:max_preview_sents], start=1):
            print(f" {i}. {textwrap.fill(_short(s, width=200), width=wrap_width)}")
        if len(naive_sents) > max_preview_sents:
            print(f"  ... (+{len(naive_sents)-max_preview_sents} more)")

        _print_sub("NLTK")
        try:
            nltk_sents = nltk_sentence_split(first_para)
            for i, s in enumerate(nltk_sents[:max_preview_sents], start=1):
                print(f" {i}. {textwrap.fill(_short(s, width=200), width=wrap_width)}")
            if len(nltk_sents) > max_preview_sents:
                print(f"  ... (+{len(nltk_sents)-max_preview_sents} more)")
        except Exception:
            print(" NLTK not available. Install: pip install nltk; python -c \"import nltk; nltk.download('punkt')\"")

        _print_sub("spaCy")
        try:
            spacy_sents = spacy_sentence_split(first_para)
            for i, s in enumerate(spacy_sents[:max_preview_sents], start=1):
                print(f" {i}. {textwrap.fill(_short(s, width=200), width=wrap_width)}")
            if len(spacy_sents) > max_preview_sents:
                print(f"  ... (+{len(spacy_sents)-max_preview_sents} more)")
        except Exception:
            print(" spaCy not available. Install: pip install spacy; python -m spacy download en_core_web_sm")


demo_splitters_pretty()


FILE: hometown.md
Paragraphs: 34 (showing first 2)

Paragraph 1:
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

Paragraph 2:
# My Hometown

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
NLTK
------------------------------------------------------------



FILE: hometown.md
Paragraphs: 34 (showing first 2)

Paragraph 1:
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

Paragraph 2:
# My Hometown

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
NLTK
------------------------------------------------------------


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radyadhewa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



FILE: hometown.md
Paragraphs: 34 (showing first 2)

Paragraph 1:
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

Paragraph 2:
# My Hometown

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
NLTK
------------------------------------------------------------


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radyadhewa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


...


FILE: hometown.md
Paragraphs: 34 (showing first 2)

Paragraph 1:
--- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

Paragraph 2:
# My Hometown

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
NLTK
------------------------------------------------------------


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radyadhewa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

------------------------------------------------------------
spaCy
------------------------------------------------------------
 1. --- title: "Hometown Overview" author: "Your Name" source: "personal" created: "2026-03-21" tags:
[hometown, local-history, guide] ---

FILE: university.md
Paragraphs: 29 (showing first 2)

Paragraph 1:
--- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"personal" created: "2026-03-21" tags: [university, sustainability, education, story] ---

Paragraph 2:
# My University — A Sustainability Story

------------------------------------------------------------
Naive sentence split
------------------------------------------------------------
 1. --- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"personal" created: "2026-03-21" tags: [university

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\radyadhewa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


 1. --- title: "My University — A Story of Studying Sustainability" author: "John Doe" source:
"personal" created: "2026-03-21" tags:
 2. [university, sustainability, education, story] ---


### LangChain's RecursiveCharacterTextSplitter  
RecursiveCharacterTextSplitter is a LangChain text-splitting utility that breaks long text into chunks by attempting to split on progressively smaller separators (paragraphs → lines → spaces → characters) until each chunk fits the target size.

#### How it Works

- Top‑down separators: It tries the largest separators first (e.g., "\n\n", then "\n", then " ", then "") to keep semantic boundaries intact.
- Recursion: If a block is still too large after splitting by a separator, it recurses to the next, smaller separator.
- Overlap: Produces overlapping chunks controlled by chunk_overlap so nearby context is preserved across chunks.
- Deterministic: Behavior is deterministic and fast, requiring no ML models.

#### Key Params


- chunk_size: target max characters per chunk.
- chunk_overlap: number of characters to overlap between consecutive chunks.
- separators: ordered list of separators tried from largest → smallest.

#### Codebase

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\radyadhewa\Storage\code\personal\AI-Engineer-RoadmapSH-Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
docs = load_markdown_docs('sample-documents')
for doc in docs:
    text = doc['text']
    print(f"\nFILE: {doc['path']}")
    print(f"Original length: {len(text)} characters")

    # Create a RecursiveCharacterTextSplitter instance
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
    
    # Split the text into chunks
    chunks = splitter.create_documents([text])
    
    print(f"Number of chunks created: {len(chunks)}")
    print(f"chunk preview: {chunks[5].page_content[:1000]}")

    # it not merely split by token count, but tries to split at natural boundaries (sentences, paragraphs) while respecting the chunk size and overlap.


FILE: sample-documents\hometown.md
Original length: 5516 characters
Number of chunks created: 45
chunk preview: and later small-scale manufacturing. The historic district preserves several 19th- and early-20th-century buildings, including the old courthouse, the market hall, and several churches.

FILE: sample-documents\university.md
Original length: 5292 characters
Number of chunks created: 49
chunk preview: Why Sustainability?
-------------------


### Semantic Chunking

### Contextual Chunking with LLMs

In [ ]:
import re

sample_text = """# Introduction
My hometown is a small city with a river and a busy market.

# History
It started as a trading town and later grew into a local center.

# Culture
People enjoy festivals, food, and community events."""

# 1) Simple sentence split
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', sample_text) if s.strip()]

# 2) Recursive character split

def recursive_split(text, chunk_size=80, overlap=20):
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

# 3) Heading-aware split

def split_by_headings(text):
    parts = re.split(r'(?m)^(#\s+.+)$', text)
    chunks = []
    current = ""
    for part in parts:
        if not part:
            continue
        if part.startswith('# '):
            if current:
                chunks.append(current.strip())
            current = part + "\n"
        else:
            current += part
    if current:
        chunks.append(current.strip())
    return chunks

print("Sentences:", sentences)
print("Recursive chunks:", recursive_split(sample_text))
print("Heading chunks:", split_by_headings(sample_text))